# SynthFace on Google Colab: train MobileFaceNet + ArcFace (pilot)

Trains a face-verification model **from scratch** on the rendered synthetic people (approach A chunks on Drive).
No pretrained face weights (CLAUDE.md). About 10% of the people are held out (never trained on) and used to measure
1:1 verification: same person vs different person.

**Every session:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.
- Uses every finished and checked chunk in `synthface_output/` (or the ones in `CHUNKS`).
- Leaves out bald women, near-duplicates and `training/exclude.json`; the list is printed and saved.
- Saves after every epoch to `synthface_output/models/<RUN_NAME>/`; running again **resumes**.
- A new `RUN_NAME` starts a new model (e.g. when more chunks are ready).

Don't run this while a render session needs the GPU time: the two compete for the same Colab quota.

## 1. Settings

In [ ]:
import time
SESSION_START = time.time()

RUN_NAME = "pilot_6chunks"      # new name = new model; same name = resume
CHUNKS = None                   # None = every finished chunk; or e.g. "p1_chunk01,p1_chunk02"
EPOCHS = 40
BATCH = 128
HOLDOUT = 0.10                  # share of people held out for testing (the same people every run)
EVAL_EVERY = 5                  # epochs between verification tests
SESSION_HOURS = 5.5
SAFETY_MINUTES = 15             # stop before the session ends (the next run resumes)

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1UhybtdRMuQroKlNt8RBiXOGVAkbDuIll"
DRIVE_DIR = None                # only if the folder is not found automatically
REPO = "https://github.com/ankitscode/-synthetic-Image-Genrator.git"
BRANCH = "master"
PROJECT = "/content/synthface"
CACHE_LOCAL = "/content/train_cache"   # images as one array on the VM disk (fast); a copy is kept on Drive

## 2. GPU, Drive, code

In [ ]:
import glob, json, os, shutil, subprocess, sys
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
from google.colab import drive
drive.mount("/content/drive")
FOLDER_ID = DRIVE_FOLDER_URL.rstrip("/").split("/")[-1].split("?")[0]


def find_drive_folder(folder_id):
    by_id = f"/content/drive/.shortcut-targets-by-id/{folder_id}"
    if os.path.isdir(by_id):
        subs = [os.path.join(by_id, d) for d in os.listdir(by_id) if os.path.isdir(os.path.join(by_id, d))]
        if len(subs) == 1:
            return subs[0]
    for root in ("/content/drive/MyDrive", "/content/drive/Shareddrives"):
        for path in [root, *glob.glob(f"{root}/*"), *glob.glob(f"{root}/*/*")]:
            try:
                if os.path.isdir(path) and os.getxattr(path, "user.drive.id").decode() == folder_id:
                    return path
            except OSError:
                pass
    return None


base = DRIVE_DIR or find_drive_folder(FOLDER_ID)
if not base:
    raise SystemExit("Shared folder not found: add a shortcut to it in My Drive or set DRIVE_DIR")
DATA = f"{base}/synthface_output"
RUN_DIR = f"{DATA}/models/{RUN_NAME}"
CACHE_DRIVE = f"{DATA}/training_cache"
print("Data:", DATA, "\nModel:", RUN_DIR)

if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
!git -C {PROJECT} log -1 --format="%h %cd %s" --date=short
!pip -q install onnx   # only for the .onnx export at the end (Apache-2.0)

## 3. What will be trained on (people per chunk, left-out people)

In [ ]:
sys.path.insert(0, PROJECT)
from pathlib import Path
from training import data as tdata
people, report = tdata.collect(Path(DATA), CHUNKS.split(",") if CHUNKS else None, HOLDOUT)
for c in report["chunks"]:
    print(f"{c['chunk']}: {c['people']} people, {c['excluded']} left out")
print("people:", report["people"], " images:", report["images"])
for e in report["excluded"]:
    print(f"  left out {e['chunk']}/{e['identity']}: {e['reason']}")
if report["skipped_incomplete"]:
    print("incomplete (no aligned images):", report["skipped_incomplete"][:10])

## 4. Train

First run: loads every aligned image from Drive once (~10 min for 18,000 images) and keeps them as one array
(copied to `training_cache/` on Drive, so later runs start in seconds). One line per epoch; verification on the
held-out people every `EVAL_EVERY` epochs.

In [ ]:
os.makedirs(CACHE_LOCAL, exist_ok=True)
os.makedirs(CACHE_DRIVE, exist_ok=True)


def cache_to_drive():
    for f in glob.glob(f"{CACHE_LOCAL}/images_{fp}_*.npy"):
        if not os.path.exists(f"{CACHE_DRIVE}/{os.path.basename(f)}"):
            shutil.copyfile(f, f"{CACHE_DRIVE}/{os.path.basename(f)}")


fp = tdata.fingerprint(people)
for f in glob.glob(f"{CACHE_DRIVE}/images_{fp}_*.npy"):          # cache from an earlier session
    if not os.path.exists(f"{CACHE_LOCAL}/{os.path.basename(f)}"):
        shutil.copyfile(f, f"{CACHE_LOCAL}/{os.path.basename(f)}")

stop_at = SESSION_START + SESSION_HOURS * 3600 - SAFETY_MINUTES * 60
cmd = [sys.executable, "-m", "training.train", "--root", DATA, "--out", RUN_DIR, "--cache-dir", CACHE_LOCAL,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--holdout", str(HOLDOUT), "--eval-every", str(EVAL_EVERY),
       "--stop-at", str(stop_at)] + (["--chunks", CHUNKS] if CHUNKS else [])
proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    if line.startswith("CACHE wrote"):
        cache_to_drive()                                            # keep it even if the session stops early
    if line.startswith(("DATA", "CACHE", "LOADED", "RESUME", "EPOCH", "STOPPED", "EXPORT", "DONE", "Traceback", "  File", "    ")) \
            or "Error" in line:
        print(line, end="", flush=True)
code = proc.wait()

cache_to_drive()
if code != 0:
    raise RuntimeError(f"training exit code {code}")

## 5. Results

**How to read them** (held-out people, never seen in training):
- **TAR@FAR=1e-3**: share of same-person pairs accepted when only 1 in 1,000 different-person pairs is wrongly accepted.
  The main number for a phone unlock / 1:1 check. Higher is better.
- **EER**: error rate where false accepts = false rejects. Lower is better.
- **AUC**: 1.0 = perfect separation.
- A big gap between `train_sample` and `holdout` = the model memorises people instead of learning faces
  (expected with few people: more chunks fix this).
These are **synthetic** people: real-photo accuracy needs a real test set.

In [ ]:
import matplotlib.pyplot as plt
m = json.load(open(f"{RUN_DIR}/metrics.json"))
hist = m["history"]
ev = [h for h in hist if "eval" in h and "holdout" in h["eval"]]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([h["epoch"] for h in hist], [h["loss"] for h in hist], label="loss")
ax[0].set_xlabel("epoch"); ax[0].set_title("training loss (margin grows in the first epochs)"); ax[0].legend()
for key, style in (("tar@far=0.001", "-o"), ("auc", "-s")):
    ax[1].plot([h["epoch"] for h in ev], [h["eval"]["holdout"][key] for h in ev], style, label=f"holdout {key}")
    ax[1].plot([h["epoch"] for h in ev], [h["eval"]["train_sample"][key] for h in ev], style, alpha=0.4, label=f"train {key}")
ax[1].plot([h["epoch"] for h in ev], [h["eval"]["holdout"]["eer"] for h in ev], "-^", label="holdout EER")
ax[1].set_xlabel("epoch"); ax[1].set_ylim(0, 1); ax[1].set_title("verification"); ax[1].legend(fontsize=8)
plt.show()
b = m["best"]
best = next(h for h in ev if h["epoch"] == b["epoch"])["eval"]
print(f"Best epoch {b['epoch']}")
for split in ("holdout", "train_sample"):
    r = best[split]
    print(f"{split:13s} AUC {r['auc']:.4f}  EER {r['eer']:.4f}  TAR@FAR=1e-2 {r['tar@far=0.01']:.4f}  "
          f"TAR@FAR=1e-3 {r['tar@far=0.001']:.4f}  best acc {r['best_accuracy']:.4f} (threshold {r['best_threshold']:.3f})  "
          f"pairs {r['pairs_genuine']}+{r['pairs_impostor']}")
print("Files:", sorted(os.listdir(RUN_DIR)))